# Gold `avaliacoes` / `clickstream_resumo`

In [0]:
import time
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalogo", "workspace")
catalogo = dbutils.widgets.get("catalogo")

dbutils.widgets.text("data_referencia_calculo", "2026-05-22")
data_referencia_param = dbutils.widgets.get("data_referencia_calculo").strip()

if data_referencia_param:
    DATA_REFERENCIA_COL = F.to_date(F.lit(data_referencia_param))
    print(f"Data de referência: {data_referencia_param}")
else:
    DATA_REFERENCIA_COL = F.current_date()
    print("Data de referência: current_date()")
    
spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print(f"Catálogo em uso: {catalogo}")

## 2. Processamento — `gold_avaliacoes`
Denormalização da tabela de avaliações com informações de clientes e produtos.

**Atenção ao Contrato:** As colunas foram renomeadas para `nome_cliente` e `categoria_produto` para garantir aderência estrita à API.

In [0]:
# Lendo as tabelas Silver necessárias para o enriquecimento
df_silver_aval = spark.table("silver.tb_avaliacoes")
df_silver_clientes = spark.table("silver.tb_clientes").select("id_cliente", "nome")
df_silver_produtos = spark.table("silver.dim_produtos").select("id_produto", "nome_produto", "categoria")

df_gold_avaliacoes = (
    df_silver_aval
    .filter(
        F.col("id_avaliacao").isNotNull()
        & F.col("id_cliente").isNotNull()
        & F.col("id_pedido").isNotNull()
        & F.col("id_produto").isNotNull()
        & F.col("sentimento").isNotNull()
        & F.col("data_avaliacao").isNotNull()
    )
    
    # Junções para denormalização 
    .join(df_silver_clientes, on="id_cliente", how="left")
    .join(df_silver_produtos, on="id_produto", how="left")
    
    # Renomeando de acordo com o contrato da gold
    .withColumnRenamed("nome", "nome_cliente")

    .select(
        "id_avaliacao",
        "id_cliente",
        "id_pedido",
        "id_produto",
        "nota_produto",
        "nota_nps",
        "recomenda",
        "comentario",
        "sentimento",
        "data_avaliacao",
        "nome_produto",
        F.coalesce(F.col("categoria"), F.lit("Sem categoria")).alias("categoria_produto"),
        "nome_cliente"
    )
)

(
    df_gold_avaliacoes.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.gold_avaliacoes")
)

print(f"gold.gold_avaliacoes criada com {df_gold_avaliacoes.count():,} registros.")

## 3. Processamento — `gold_clickstream_resumo`
Resumo diário do comportamento digital por cliente, criando uma série temporal (uma linha por cliente/dia) para o CRM e Agente de IA.

In [0]:
df_silver_click = spark.table("silver.tb_clickstream")

df_gold_clickstream_resumo = (
    df_silver_click
    .filter(F.col("id_cliente").isNotNull())
    
    # Ajuste de nome da coluna de data para compor a PK 
    .withColumnRenamed("data_evento", "data")
    
    # Agrupamento diário por cliente
    .groupBy("id_cliente", "data")
    .agg(
        F.count("*").alias("qtd_eventos"),
        F.countDistinct("id_sessao").alias("qtd_sessoes"),
        
        # Mapeamento para os nomes exato
        F.sum(F.when(F.col("tipo_evento") == "visualizacao_pagina", 1).otherwise(0)).alias("qtd_page_view"),
        F.sum(F.when(F.col("tipo_evento") == "busca", 1).otherwise(0)).alias("qtd_search"),
        F.sum(F.when(F.col("tipo_evento") == "visualizacao_produto", 1).otherwise(0)).alias("qtd_click"),
        F.sum(F.when(F.col("tipo_evento") == "adicao_carrinho", 1).otherwise(0)).alias("qtd_add_to_cart"),
        F.sum(F.when(F.col("tipo_evento") == "abandono_carrinho", 1).otherwise(0)).alias("qtd_abandon_cart"),
        F.sum(F.when(F.col("tipo_evento") == "compra", 1).otherwise(0)).alias("qtd_purchase"),
        
        # uso da função mode() para encontrar os dominantes do dia
        F.expr("mode(canal)").alias("canal_principal"),
        F.expr("mode(dispositivo)").alias("dispositivo_principal"),
        # Evita null quando o cliente/dia tem eventos, mas nenhum tempo válido
        F.coalesce(
            F.sum("tempo_pagina_seg"),
            F.lit(0)
        ).cast("long").alias("tempo_total_segundos")
    )
    .withColumn("data_referencia_calculo", DATA_REFERENCIA_COL)
)

(
    df_gold_clickstream_resumo.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.gold_clickstream_resumo")
)

print(f"gold.gold_clickstream_resumo criada com {df_gold_clickstream_resumo.count():,} linhas (cliente/dia).")

## 4. Validação Técnica (Quality Gate Gold)
Garante que as tabelas existem, não estão vazias e possuem o esquema esperado antes de disponibilizar para a aplicação.

In [0]:
print("\nIniciando Quality Gate da camada Gold...")

tabelas_gold = ["gold_avaliacoes", "gold_clickstream_resumo"]
falhas = []

for tabela in tabelas_gold:
    nome_completo = f"gold.{tabela}"
    
    if not spark.catalog.tableExists(nome_completo):
        print(f"[ERRO] {nome_completo} não encontrada no catálogo.")
        falhas.append(f"{nome_completo} ausente")
        continue

    df = spark.table(nome_completo)
    total = df.count()
    
    if total == 0:
        print(f"[ERRO] {nome_completo} está vazia após processamento.")
        falhas.append(f"{nome_completo} vazia")
        continue
        
    print(f"[OK] {nome_completo} validada com sucesso ({total:,} registros).")

if falhas:
    raise Exception("Quality Gate da Gold falhou:\n- " + "\n- ".join(falhas))

print("\nCamada Gold validada.")